<a href="https://colab.research.google.com/github/alexaK88/Fun_tasks_for_the_weekend/blob/main/word2vec_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import numpy as np
from tqdm import tqdm
from collections import Counter

In [4]:
text_path = "/content/drive/MyDrive/star_rover.txt"

In [5]:
def build_vocab(tokens, min_count=2):

    counts = Counter(tokens)
    vocab = [w for w, c in counts.items() if c >= min_count]

    word2ix = {w:i for i, w in enumerate(vocab)}
    idx2word = {i:w for w, i in word2ix.items()}

    return word2ix, idx2word

In [6]:
class NegativeSampler:
    def __init__(self, word_freq, power=0.75):
        freq = np.array(word_freq) ** power
        self.prob = freq / freq.sum()

    def sample(self, K):
        return np.random.choice(len(self.prob), size=K, p=self.prob)

In [7]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


class Word2VecSGNS:

    def __init__(self, vocab_size, embed_dim=100, lr=0.025):

        self.V = vocab_size
        self.D = embed_dim
        self.lr = lr

        # input embeddings
        self.W_in = np.random.randn(self.V, self.D) * 0.01

        # output embeddings
        self.W_out = np.random.randn(self.V, self.D) * 0.01

    def train_pair(self, center, context, negatives):
        """
        One SGD update step.
        """

        v_c = self.W_in[center]
        v_o = self.W_out[context]

        neg_vectors = self.W_out[negatives]

        # ---------- Forward ----------
        pos_score = np.dot(v_o, v_c)
        neg_scores = neg_vectors @ v_c

        pos_sig = sigmoid(pos_score)
        neg_sig = sigmoid(neg_scores)

        # Loss (optional for logging)
        loss = -np.log(pos_sig + 1e-10) \
               -np.sum(np.log(1 - neg_sig + 1e-10))

        # ---------- Gradients ----------

        # positive gradient
        grad_pos = pos_sig - 1

        # negative gradients
        grad_neg = neg_sig

        # gradient wrt center embedding
        grad_center = (
            grad_pos * v_o +
            np.sum(grad_neg[:, None] * neg_vectors, axis=0)
        )

        # ---------- Updates ----------

        # update output embeddings
        self.W_out[context] -= self.lr * grad_pos * v_c

        self.W_out[negatives] -= (
            self.lr * grad_neg[:, None] * v_c
        )

        # update center embedding
        self.W_in[center] -= self.lr * grad_center

        return loss


In [8]:
WINDOW = 2
NEG_SAMPLES = 5
EPOCHS = 3

tokens = open(text_path).read().lower().split()

vocab = list(set(tokens))
word2idx = {w:i for i,w in enumerate(vocab)}
encoded = np.array([word2idx[w] for w in tokens])

freq = np.bincount(encoded)
sampler = NegativeSampler(freq)

model = Word2VecSGNS(len(vocab), embed_dim=100)

for epoch in range(EPOCHS):
    total_loss = 0

    for i in tqdm(range(WINDOW, len(encoded)-WINDOW)):

        center = encoded[i]
        context_words = encoded[i-WINDOW:i+WINDOW+1]

        for context in context_words:
            if context == center:
                continue

            negatives = sampler.sample(NEG_SAMPLES)

            loss = model.train_pair(center, context, negatives)
            total_loss += loss

    print("Epoch loss:", total_loss)

100%|██████████| 106618/106618 [02:11<00:00, 810.68it/s]


Epoch loss: 1425536.6580389587


100%|██████████| 106618/106618 [02:12<00:00, 807.06it/s]


Epoch loss: 1190472.8616531438


100%|██████████| 106618/106618 [02:11<00:00, 807.72it/s]

Epoch loss: 1103900.856565957
